In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

# Filter rows where 'time' column is 'before standard treatment'
metadata_df = metadata_df[metadata_df['time'] == 'before standard treatment']
print(metadata_df.shape)
metadata_df


(22, 10)


,sample_title,patient id,age (at surgery),Sex,tumor grade,overall survival,treatment,tissue,tumor type,time
samples,,,,,,,,,,
GSM5690404,Ge699P7,Ge699,57,M,IV,17 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690405,Ge699P8,Ge699,57,M,IV,17 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690407,Ge755P6,Ge755,34,F,IV,18 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690409,Ge778P7,Ge778,43,M,IV,35 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690410,Ge822P8,Ge822,71,M,IV,5 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690411,Ge835P1,Ge835,50,M,IV,10 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690413,Ge849,Ge849,57,F,IV,18 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690416,Ge882,Ge882,67,M,IV,24 months,radiation and TMZ,brain,glioma,before standard treatment
GSM5690417,Ge893,Ge893,56,M,IV,41 months,radiation and TMZ,brain,glioma,before standard treatment


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = df_gene_annotations.set_index('GeneID')['Symbol'].to_dict()
df_combined = df_combined.rename(columns=gene_mapping)


# Drop non-unique rows in df_combined based on 'patient id', keeping the first occurrence
# Its not clear which samples should we select and what are the QC metrics
df_combined = df_combined.drop_duplicates(subset='patient id', keep='first')


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_2968/2018102994.py:16: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,patient id,age (at surgery),Sex,tumor grade,overall survival,treatment,tissue,tumor type,time,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM5690404,Ge699P7,Ge699,57,M,IV,17 months,radiation and TMZ,brain,glioma,before standard treatment,...,2589,1,1,0,2039,515,10,965,2,3
GSM5690407,Ge755P6,Ge755,34,F,IV,18 months,radiation and TMZ,brain,glioma,before standard treatment,...,2465,1,1,1,1939,633,5,662,1,4
GSM5690409,Ge778P7,Ge778,43,M,IV,35 months,radiation and TMZ,brain,glioma,before standard treatment,...,1612,1,1,1,1243,307,31,557,5,2
GSM5690410,Ge822P8,Ge822,71,M,IV,5 months,radiation and TMZ,brain,glioma,before standard treatment,...,5068,1,0,2,4276,977,84,1879,1,2
GSM5690411,Ge835P1,Ge835,50,M,IV,10 months,radiation and TMZ,brain,glioma,before standard treatment,...,855,0,0,1,985,226,27,390,1,2
GSM5690413,Ge849,Ge849,57,F,IV,18 months,radiation and TMZ,brain,glioma,before standard treatment,...,5901,6,4,3,5176,1568,132,1537,2,6
GSM5690416,Ge882,Ge882,67,M,IV,24 months,radiation and TMZ,brain,glioma,before standard treatment,...,2906,3,4,4,2457,583,65,969,1,2
GSM5690417,Ge893,Ge893,56,M,IV,41 months,radiation and TMZ,brain,glioma,before standard treatment,...,3897,1,1,1,2742,566,53,1549,2,6
GSM5690419,Ge901,Ge901,23,F,III,37 months,radiation and TMZ,brain,glioma,before standard treatment,...,7870,6,6,6,6941,1799,180,3297,9,2


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
